In [ ]:
from gensim.test.utils import datapath
import gensim.downloader as api
from gensim.models import KeyedVectors
from scipy.stats import spearmanr
import numpy as np

In [ ]:
# Load the pre-trained embeddings using gensim

glove = api.load("glove-twitter-200")
fasttext = api.load("fasttext-wiki-news-subwords-300")

[==================================================] 100.0% 958.5/958.4MB downloaded


In [ ]:
print("GloVe loaded: vocab-size: ", len(glove.key_to_index))
print("FastText loaded: vocab-size: ", len(fasttext.key_to_index))

GloVe loaded: vocab-size:  1193514
FastText loaded: vocab-size:  999999


## Use Case: Word Similarity Task on WordSim-353 dataset

In [ ]:
wordsim_path = datapath("/content/wordsim353.tsv")

In [ ]:
print("Evaluation on WordSim 353 dataset")

glove_wordsim = glove.evaluate_word_pairs(wordsim_path)
fasttext_wordsim = fasttext.evaluate_word_pairs(wordsim_path)

print("GloVe Pearson correlation: ", glove_wordsim[0][0])
print("GloVe Spearman correlation: ", glove_wordsim[1].correlation)
print("GloVe OOV ratio: ", glove_wordsim[2])

print("FastText Pearson correlation: ", fasttext_wordsim[0][0])
print("FastText Spearman correlation: ", fasttext_wordsim[1].correlation)
print("FastText OOV ratio: ", fasttext_wordsim[2])

Evaluation on WordSim 353 dataset
GloVe Pearson correlation:  0.5269937276503416
GloVe Spearman correlation:  0.5211594592905648
GloVe OOV ratio:  2.26628895184136
FastText Pearson correlation:  0.6064465554824066
FastText Spearman correlation:  0.5959362724389288
FastText OOV ratio:  0.0


In [ ]:
test_words = ['king', 'queen', 'apple', 'apples', 'run', 'running', 'happyness', 'data', 'awesomme', 'covid19']

print("===Nearest Neighbors for selected words===")

for word in test_words:
  print(f"\nWord: '{word}'")

  if word in glove:
    glove_neighbors = [w for w,_ in glove.most_similar(word, topn=5)]
    print("Glove:", glove_neighbors)
  else:
    print("Glove: OOV(not in vocabulary)")

  if word in fasttext:
    fasttext_neighbors = [w for w, _ in fasttext.most_similar(word, topn=5)]
    print("FastText:", fasttext_neighbors)
  else:
    print("FastText: OOV(not in vocabulary)")

===Nearest Neighbors for selected words===

Word: 'king'
Glove: ['prince', 'queen', 'aka', 'kings', 'burger']
FastText: ['king-', 'boy-king', 'queen', 'prince', 'kings']

Word: 'queen'
Glove: ['princess', 'king', 'queens', 'diva', 'lady']
FastText: ['queens', 'king', 'queendom', 'queen-', 'queening']

Word: 'apple'
Glove: ['samsung', 'microsoft', 'iphone', 'google', 'blackberry']
FastText: ['apples', 'pear', 'peach', 'fruit', 'apple-']

Word: 'apples'
Glove: ['oranges', 'pears', 'grapes', 'bananas', 'strawberries']
FastText: ['apple', 'oranges', 'pears', 'peaches', 'Apples']

Word: 'run'
Glove: ['running', 'walk', 'runs', 'way', 'ran']
FastText: ['running', 'ran', 'runnning', 'runs', 'trun']

Word: 'running'
Glove: ['run', 'walking', 'ran', 'around', 'walk']
FastText: ['runnning', 'run', 'running-', 'runing', 'runnign']

Word: 'happyness'
Glove: ['hapiness', 'happines', 'pursuit', 'happiness', 'happieness']
FastText: ['hapiness', 'saddness', 'happiness', 'crazyness', 'maddness']

Word:

In [ ]:
# Assuming you have already loaded GloVe and FastText models
king_vector = glove['king']
man_vector = glove['man']
woman_vector = glove['woman']

# Calculate the result for GloVe
result_glove = king_vector - man_vector + woman_vector

# Find the nearest word
nearest_word_glove = glove.similar_by_vector(result_glove, topn=1)

# Repeat for FastText
king_vector_ft = fasttext['king']
man_vector_ft = fasttext['man']
woman_vector_ft = fasttext['woman']

result_fasttext = king_vector_ft - man_vector_ft + woman_vector_ft
nearest_word_fasttext = fasttext.similar_by_vector(result_fasttext, topn=1)

# Print results
print("GloVe Result: ", nearest_word_glove)
print("FastText Result: ", nearest_word_fasttext)

GloVe Result:  [('king', 0.6921419501304626)]
FastText Result:  [('king', 0.8185327053070068)]


In [ ]:
import numpy as np

# Calculate the vector using GloVe
king_vector = glove['king']
man_vector = glove['man']
woman_vector = glove['woman']

result_glove = king_vector - man_vector + woman_vector

# Get the vector for "queen"
queen_vector = glove['queen']

# Calculate the cosine similarity manually
similarity_score_glove = np.dot(result_glove, queen_vector) / (np.linalg.norm(result_glove) * np.linalg.norm(queen_vector))

print("GloVe similarity score for 'queen':", similarity_score_glove)

# Repeat for FastText
king_vector_ft = fasttext['king']
man_vector_ft = fasttext['man']
woman_vector_ft = fasttext['woman']

result_fasttext = king_vector_ft - man_vector_ft + woman_vector_ft

# Get the vector for "queen" in FastText
queen_vector_ft = fasttext['queen']

# Calculate the cosine similarity for FastText
similarity_score_fasttext = np.dot(result_fasttext, queen_vector_ft) / (np.linalg.norm(result_fasttext) * np.linalg.norm(queen_vector_ft))

print("FastText similarity score for 'queen':", similarity_score_fasttext)

GloVe similarity score for 'queen': 0.65601796
FastText similarity score for 'queen': 0.75703365


- FastText has given better results, because they are better in handling rare words, misspellings, morphology etc.
- Generally preferred for most moder/general NLP tasks

## Recommendation

- For general use (news/wiki/formal text), go for FastText
- For Twitter, gloVe
- For social media related use cases, gloVe